# Module 7: Spatial Hotspot Detection

**Task 7**: Detect spatial hotspots of change and relate them to physical (terrain) and human factors (State level).

**Deliverables**:
- Grid shapefile with Gi* z-scores and p-values
- Hotspot/coldspot maps
- Terrain correlation scatter plots
- Road density correlation scatter plots
- Regression results summary

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterstats import zonal_stats

from config import (
    STATES, YEARS, LULC_CLASSES,
    PROCESSED_DIR, FIGURES_DIR, LULC_DIR, DEM_DIR, ROADS_DIR,
    BOUNDARIES_DIR, GEE_EXPORTS_DIR,
    GRID_CELL_SIZE_M, TIER2_SCALE, PIXEL_AREA_KM2,
)
from scripts.hotspot import (
    create_local_square_grid,
    compute_getis_ord_gi_star,
    add_terrain_factors,
    add_human_factors,
    run_regression_analysis,
)
from scripts.visualization import save_figure, plot_hotspot_map
from scripts.lulc_utils import save_composition_csv

print('Imports successful!')

## 7.1: Create Change Intensity Grid

In [ ]:
change_grids = {}

for state_key, state_cfg in STATES.items():
    print(f'Creating {GRID_CELL_SIZE_M/1000:.0f} km grid for {state_cfg["name"]}...')
    
    # Load state boundary
    boundary_file = BOUNDARIES_DIR / f'{state_key}_district_boundaries.geojson'
    if not boundary_file.exists():
        print(f'  ⚠️ Boundary not found')
        continue
    
    state_gdf = gpd.read_file(boundary_file).dissolve()  # Merge all districts
    
    # Create grid
    grid = create_local_square_grid(
        state_gdf, cell_size_m=GRID_CELL_SIZE_M, crs_utm=state_cfg['utm_epsg']
    )
    print(f'  Grid: {len(grid)} cells')
    
    # Compute change intensity from 2016→2025 rasters
    raster_2016 = LULC_DIR / f'{state_key}_30m_2016.tif'
    raster_2025 = LULC_DIR / f'{state_key}_30m_2025.tif'
    
    if not raster_2016.exists() or not raster_2025.exists():
        print(f'  ⚠️ LULC rasters not found for change computation')
        # Try loading from GEE CSV exports
        grid_csv = GEE_EXPORTS_DIR / f'{state_key}_grid_change_intensity.csv'
        if grid_csv.exists():
            print(f'  Loading pre-computed grid from {grid_csv.name}')
            grid_data = pd.read_csv(grid_csv)
            grid['change_intensity'] = grid_data['change_intensity'].values[:len(grid)]
        else:
            print(f'  ⚠️ No change data available — skipping')
            continue
    else:
        print(f'  Computing change intensity from rasters...')
        with rasterio.open(raster_2016) as src16, rasterio.open(raster_2025) as src25:
            # Zonal stats for 2016 and 2025
            stats_16 = zonal_stats(grid, str(raster_2016), categorical=True, nodata=255)
            stats_25 = zonal_stats(grid, str(raster_2025), categorical=True, nodata=255)
        
        intensities = []
        for s16, s25 in zip(stats_16, stats_25):
            total = sum(s16.values()) if s16 else 1
            # Count pixels that changed class
            changed = 0
            all_classes = set(list(s16.keys()) + list(s25.keys()))
            for cls in all_classes:
                c16 = s16.get(cls, 0)
                c25 = s25.get(cls, 0)
                changed += abs(c25 - c16)
            # Divide by 2 since each change is counted twice (loss + gain)
            intensity = (changed / 2) / total if total > 0 else 0
            intensities.append(min(intensity, 1.0))
        
        grid['change_intensity'] = intensities
    
    change_grids[state_key] = grid
    print(f'  Mean intensity: {grid["change_intensity"].mean():.3f}')
    print(f'  Max intensity: {grid["change_intensity"].max():.3f}')

## 7.2: Getis-Ord Gi* Analysis

In [ ]:
hotspot_grids = {}

for state_key, state_cfg in STATES.items():
    if state_key not in change_grids:
        continue
    
    grid = change_grids[state_key]
    print(f'Running Gi* for {state_cfg["name"]}...')
    
    try:
        result = compute_getis_ord_gi_star(grid, 'change_intensity')
        hotspot_grids[state_key] = result
        
        # Summary
        counts = result['hotspot_class'].value_counts()
        print(f'  Results:')
        for cls, cnt in counts.items():
            print(f'    {cls}: {cnt} cells')
        
        # Save
        outdir = PROCESSED_DIR / 'hotspots'
        outdir.mkdir(parents=True, exist_ok=True)
        result.to_file(outdir / f'{state_key}_hotspot_grid.geojson', driver='GeoJSON')
        
    except Exception as e:
        print(f'  ❌ Error: {e}')
        hotspot_grids[state_key] = grid

In [ ]:
# Plot hotspot maps
for state_key, state_cfg in STATES.items():
    if state_key in hotspot_grids and 'hotspot_class' in hotspot_grids[state_key].columns:
        plot_hotspot_map(hotspot_grids[state_key], state_cfg['name'])

print('✅ Hotspot maps generated.')

## 7.3: Relate to Terrain Factors

In [ ]:
for state_key, state_cfg in STATES.items():
    if state_key not in hotspot_grids:
        continue
    
    grid = hotspot_grids[state_key]
    dem_path = DEM_DIR / f'{state_key}_elevation.tif'
    slope_path = DEM_DIR / f'{state_key}_slope.tif'
    
    if dem_path.exists() or slope_path.exists():
        print(f'Adding terrain factors for {state_cfg["name"]}...')
        grid = add_terrain_factors(
            grid,
            dem_path=str(dem_path) if dem_path.exists() else None,
            slope_path=str(slope_path) if slope_path.exists() else None
        )
        hotspot_grids[state_key] = grid
    
    # Scatter: slope vs change intensity
    if 'slope_mean' in grid.columns:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        ax1.scatter(grid['slope_mean'], grid['change_intensity'],
                    alpha=0.3, s=10, c='#d73027')
        ax1.set_xlabel('Mean Slope (degrees)')
        ax1.set_ylabel('Change Intensity')
        ax1.set_title('Slope vs. Change Intensity')
        ax1.grid(True, alpha=0.3)
        
        if 'elevation_mean' in grid.columns:
            ax2.scatter(grid['elevation_mean'], grid['change_intensity'],
                        alpha=0.3, s=10, c='#4575b4')
            ax2.set_xlabel('Mean Elevation (m)')
            ax2.set_ylabel('Change Intensity')
            ax2.set_title('Elevation vs. Change Intensity')
            ax2.grid(True, alpha=0.3)
        
        fig.suptitle(f'Terrain Factors — {state_cfg["name"]}', fontsize=13)
        fig.tight_layout()
        save_figure(fig, f'{state_key}_terrain_correlation.png', 'hotspot_maps')
        plt.show()

## 7.4: Relate to Human Factors

In [ ]:
for state_key, state_cfg in STATES.items():
    if state_key not in hotspot_grids:
        continue
    
    grid = hotspot_grids[state_key]
    
    # Load roads
    road_file = ROADS_DIR / f'{state_key}_major_roads.gpkg'
    roads_gdf = gpd.read_file(road_file) if road_file.exists() else None
    
    print(f'Adding human factors for {state_cfg["name"]}...')
    grid = add_human_factors(
        grid, roads_gdf, state_cfg['cities'], state_cfg['utm_epsg']
    )
    hotspot_grids[state_key] = grid
    
    # Scatter plots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    if 'road_density_km_per_km2' in grid.columns:
        ax1.scatter(grid['road_density_km_per_km2'], grid['change_intensity'],
                    alpha=0.3, s=10, c='#e6550d')
        ax1.set_xlabel('Road Density (km/km²)')
        ax1.set_ylabel('Change Intensity')
        ax1.set_title('Road Density vs. Change')
        ax1.grid(True, alpha=0.3)
    
    if 'distance_to_city_km' in grid.columns:
        ax2.scatter(grid['distance_to_city_km'], grid['change_intensity'],
                    alpha=0.3, s=10, c='#756bb1')
        ax2.set_xlabel('Distance to Nearest City (km)')
        ax2.set_ylabel('Change Intensity')
        ax2.set_title('City Proximity vs. Change')
        ax2.grid(True, alpha=0.3)
    
    fig.suptitle(f'Human Factors — {state_cfg["name"]}', fontsize=13)
    fig.tight_layout()
    save_figure(fig, f'{state_key}_human_factors.png', 'hotspot_maps')
    plt.show()

## 7.5: Multiple Regression

In [ ]:
for state_key, state_cfg in STATES.items():
    if state_key not in hotspot_grids:
        continue
    
    grid = hotspot_grids[state_key]
    
    print(f'\nRegression Analysis — {state_cfg["name"]}')
    print('=' * 60)
    print('Model: change_intensity ~ slope + elevation + road_density + distance_to_city')
    print()
    
    results = run_regression_analysis(grid)
    
    if 'error' in results:
        print(f'  ❌ {results["error"]}')
    else:
        print(f'  R²: {results["r_squared"]:.4f}')
        print(f'  Adj R²: {results["adj_r_squared"]:.4f}')
        print(f'  N: {results["n_observations"]}')
        print(f'\n  Coefficients:')
        for name, coefs in results['coefficients'].items():
            sig = '***' if coefs['p_value'] < 0.01 else '**' if coefs['p_value'] < 0.05 else '*' if coefs['p_value'] < 0.1 else ''
            print(f'    {name:20s}: β={coefs["estimate"]:+.6f}  '
                  f'SE={coefs["std_error"]:.6f}  p={coefs["p_value"]:.4f} {sig}')
    
    # Save grid with all factors
    outdir = PROCESSED_DIR / 'hotspots'
    grid.to_file(outdir / f'{state_key}_hotspot_grid_full.geojson', driver='GeoJSON')

print('\n✅ Module 7 complete.')

---
## Summary
- ✅ 5 km grid with change intensity per cell
- ✅ Getis-Ord Gi* hotspot/coldspot classification
- ✅ Terrain correlation (slope, elevation)
- ✅ Human factor correlation (road density, city proximity)
- ✅ Multiple regression model

**Next**: `08_ranking_index.ipynb`